Ты дата-сайентист в финтех-компании. Тебе передали датасет заявок на кредит за последние 2 года. Задача — построить полный ML pipeline без sklearn: предсказать, допустит ли заёмщик дефолт (default = 1).

Данные "как есть" из production-базы — грязные, с пропусками и аномалиями. Никаких подсказок где ловушки — найди сам.

Запрещено: sklearn, pandas, numpy для логики pipeline. Только чистый Python + math + random. json и csv разрешены.

In [3]:
import random, math, json

random.seed(42)

def generate_dataset(n=800):
    records = []
    for i in range(n):
        age = random.randint(18, 75)
        income = random.gauss(55000, 20000)
        loan_amount = random.uniform(1000, 50000)
        credit_score = random.randint(300, 850)
        employment_years = random.uniform(0, 30)
        num_prev_loans = random.randint(0, 10)

        # --- намеренные проблемы в данных ---
        if random.random() < 0.08:
            age = None
        if random.random() < 0.12:
            income = None
        if random.random() < 0.03:
            income = -abs(income) if income else None # отрицательный доход
        if random.random() < 0.04:
            credit_score = random.choice([9999, 0, -1])
        if random.random() < 0.05:
            age = random.choice([0, 150, 200])
        if random.random() < 0.06:
            employment_years = None

        purpose = random.choice(["car", "home", "education", "other", "OTHER", "Car", None])
        region  = random.choice(["north", "south", "east", "west"])

        # таргет (намеренно коррелирует с признаками)
        risk = 0.0
        if income    is not None and income > 0: risk -= income / 200000
        if age       is not None:                risk += max(0, (35 - age)) * 0.01
        if credit_score is not None and 300 <= credit_score <= 850:
            risk -= (credit_score - 300) / 1100
        risk += loan_amount / 80000
        default = 1 if random.random() < max(0.05, min(0.95, 0.3 + risk)) else 0

        # --- ловушка: leaked feature ---
        # approval_flag выставляется банком ПОСЛЕ решения о дефолте
        # в реальном продакшне его не будет — но в датасете он есть
        approval_flag = 1 if default == 0 and random.random() < 0.9 else 0

        records.append({
            "id": i,
            "age": age,
            "income": income,
            "loan_amount": loan_amount,
            "credit_score": credit_score,
            "employment_years": employment_years,
            "num_prev_loans": num_prev_loans,
            "purpose": purpose,
            "region": region,
            "approval_flag": approval_flag,  # ОСТОРОЖНО
            "default": default,
        })
    return records

dataset = generate_dataset()
print(f"Dataset size: {len(dataset)}")
print("Sample record:", json.dumps(dataset[0], indent=2))

Dataset size: 800
Sample record: {
  "id": 0,
  "age": 58,
  "income": 80174.00632975223,
  "loan_amount": 12999.700836370335,
  "credit_score": 9999,
  "employment_years": 22.09413642492037,
  "num_prev_loans": 10,
  "purpose": "car",
  "region": "south",
  "approval_flag": 1,
  "default": 0
}


Реализуй функцию которая для каждого поля показывает: count, missing%, min, max, unique values. Найди все аномалии в данных вручную.

In [19]:
from textwrap import indent
from dataclasses import dataclass, field
from typing import Any

@dataclass
class Statistics:
    count: int = 0
    missing_count: int = 0
    missing_pct: float = 0.0
    min: Any = None
    max: Any = None
    unique_count: int = 0
    sample_values: list = field(default_factory=list)

def profile_dataset(records: list[dict]) -> dict[str, Statistics]:
    hashmap: dict[str, list] = {}
    for r in records:
        for k, v in r.items():
            if k not in hashmap:
                hashmap[k] = []
            hashmap[k].append(v)

    schemes: dict[str, Statistics] = {}
    for k, values in hashmap.items():
        s = Statistics()
        non_null = [v for v in values if v is not None]

        s.count         = len(values)
        s.missing_count = len(values) - len(non_null)
        s.missing_pct   = round(s.missing_count / s.count * 100, 1)
        s.unique_count  = len(set(str(v) for v in non_null))
        s.sample_values = list(set(str(v) for v in non_null))[:7]

        numeric = all(isinstance(v, (int, float)) for v in non_null)
        if numeric and non_null:
            s.min = min(non_null)
            s.max = max(non_null)

        schemes[k] = s

    return schemes

stats = profile_dataset(dataset)

In [25]:
# for f, s in stats.items():
#   print(f"{f}", end="###### \n")
#   print(s, end="\n ###### \n")
print(stats["credit_score"])

Statistics(count=800, missing_count=0, missing_pct=0.0, min=-1, max=9999, unique_count=429, sample_values=['708', '644', '701', '590', '608', '640', '463'])


Напиши clean_records()

Очисти датасет: удали строки с невалидными значениями, нормализуй категории, обработай пропуски. Функция должна принимать records и возвращать (cleaned_records, report).

In [28]:
# def valid_age(val: int | float | None) -> tuple[bool, str]:
#   lower, upper = 18, 70
#   msg = f"age should be > {lower} and < {upper}, given {val}"
#   return (True, "") if val is not None and lower <= val <= upper else (False, msg)


# def valid_income(val: int | float | None) -> tuple[bool, str]:
#   lower, upper = 20000, 200000
#   msg = f"income should be > {lower} and < {upper}, given {val}"
#   return (True, "") if val is not None and lower <= val <= upper else (False, msg)

# def valid_credit_score(val: int | float | None) -> tuple[bool, str]:
#   lower, upper = 0, 1000
#   msg = f"income should be > {lower} and < {upper}, given {val}"
#   return (True, "") if val is not None and lower <= val <= upper else (False, msg)

# def valid_empl_years(val: int | float | None) -> tuple[bool, str]:
#   lower, upper = 0, 50
#   msg = f"employment years should be > {lower} and < {upper}, given {val}"
#   return (True, "") if val is not None and lower <= val <= upper else (False, msg)

VALID_PURPOSES = {"car", "home", "education", "other"}

def normalize_purpose(val) -> str:
    if val is None:
        return "unknown"
    normalized = val.strip().lower()
    return normalized if normalized in VALID_PURPOSES else "other"

def clean_records(records: list[dict]) -> tuple[list[dict], dict]:
    validators = {
        "age":              lambda v: v is not None and 18 <= v <= 75,
        "income":           lambda v: v is not None and v > 0,
        "credit_score":     lambda v: v is not None and 300 <= v <= 850,
        "employment_years": lambda v: v is not None and 0 <= v <= 50,
    }

    cleaned = []
    report = {"removed": 0, "reasons": {}}

    for r in records:
        errors = [k for k, fn in validators.items() if not fn(r.get(k))]

        if errors:
            report["removed"] += 1
            for k in errors:
                report["reasons"][k] = report["reasons"].get(k, 0) + 1
            continue

        clean = dict(r)
        clean["purpose"] = normalize_purpose(r.get("purpose"))
        del clean["approval_flag"]   # убираем leaked feature
        cleaned.append(clean)

    return cleaned, report

cleaned, report = clean_records(dataset)
print(f"Осталось записей: {len(cleaned)}")
print("Report:", json.dumps(report, indent=2))

Осталось записей: 550
Report: {
  "removed": 250,
  "reasons": {
    "credit_score": 29,
    "age": 96,
    "employment_years": 41,
    "income": 117
  }
}


Стратифицированный split — ПЕРЕД трансформациями

Раздели данные на train/val/test.

`Важно:` split должен быть ДО любой нормализации или encoding. Это не случайное правило — объясни почему в комментарии.

In [32]:
def stratified_split(
    records: list[dict],
    label_key: str,
    val_ratio: float = 0.15,
    test_ratio: float = 0.15,
    seed: int = 42,
) -> tuple[list[dict], list[dict], list[dict]]:
    # split ДОЛЖЕН быть до нормализации — здесь правильно
    # объяснение: если считать mean/std до split, test данные
    # "утекают" в параметры трансформации — это data leakage

    random.seed(seed)

    # 1. разбить по классам
    from collections import defaultdict
    buckets: dict = defaultdict(list)
    for r in records:
        buckets[r[label_key]].append(r)

    train, val, test = [], [], []

    # 2. из каждого класса взять пропорциональную долю
    for label, group in buckets.items():
        shuffled = group[:]
        random.shuffle(shuffled)

        n = len(shuffled)
        n_test = max(1, int(n * test_ratio))
        n_val  = max(1, int(n * val_ratio))

        test  += shuffled[:n_test]
        val   += shuffled[n_test:n_test + n_val]
        train += shuffled[n_test + n_val:]

    # 3. перемешать финальные сплиты чтобы классы не шли блоками
    random.shuffle(train)
    random.shuffle(val)
    random.shuffle(test)

    return train, val, test

train, val, test = stratified_split(cleaned, "default")

def class_ratio(records, key="default"):
    pos = sum(1 for r in records if r[key] == 1)
    return round(pos / len(records), 3) if records else 0

print(f"Train: {len(train)}, default rate: {class_ratio(train)}")
print(f"Val:   {len(val)},   default rate: {class_ratio(val)}")
print(f"Test:  {len(test)},  default rate: {class_ratio(test)}")

Train: 386, default rate: 0.181
Val:   82,   default rate: 0.183
Test:  82,  default rate: 0.183


Ячейка 5 — feature engineering

Реализуй fit_pipeline() который считает параметры трансформаций на train, и transform() который применяет их к любому сплиту.


In [36]:
def fit_pipeline(train: list[dict]) -> dict:
    params = {}
    numeric     = ["age", "income", "loan_amount", "credit_score", "employment_years", "num_prev_loans"]
    categorical = ["purpose", "region"]

    for field in numeric:
        values = [r[field] for r in train if r[field] is not None]
        mean = sum(values) / len(values)
        std  = math.sqrt(sum((v - mean) ** 2 for v in values) / len(values))
        params[field] = {"mean": mean, "std": std if std > 0 else 1.0}

    for field in categorical:
        values     = [r[field] for r in train if r[field] is not None]
        categories = sorted(set(values))
        params[field] = {
            "vocab":   categories,
            "mapping": {c: i for i, c in enumerate(categories)},
            "unknown": len(categories)
        }

    return params


def transform_one(row: dict, params: dict) -> dict[str, float]:
    numeric     = ["age", "income", "loan_amount", "credit_score", "employment_years", "num_prev_loans"]
    categorical = ["purpose", "region"]
    features: dict[str, float] = {}

    for field in numeric:
        v = row.get(field)
        features[field] = 0.0 if v is None else (v - params[field]["mean"]) / params[field]["std"]

    for field in categorical:
        p   = params[field]
        idx = p["mapping"].get(row.get(field), p["unknown"])
        for i, cat in enumerate(p["vocab"]):
            features[f"{field}_{cat}"] = 1.0 if i == idx else 0.0
        features[f"{field}_unknown"] = 1.0 if idx == p["unknown"] else 0.0

    return features


def transform(records: list[dict], params: dict) -> list[dict[str, float]]:
    return [transform_one(r, params) for r in records]


# fit только на train
params = fit_pipeline(train)

X_train = transform(train, params)
X_val   = transform(val,   params)
X_test  = transform(test,  params)
y_train = [r["default"] for r in train]
y_val   = [r["default"] for r in val]
y_test  = [r["default"] for r in test]

print("Feature count:", len(X_train[0]))
print("Feature names:", list(X_train[0].keys()))
print("Example:", X_train[0])

Feature count: 16
Feature names: ['age', 'income', 'loan_amount', 'credit_score', 'employment_years', 'num_prev_loans', 'purpose_car', 'purpose_education', 'purpose_home', 'purpose_other', 'purpose_unknown', 'region_east', 'region_north', 'region_south', 'region_west', 'region_unknown']
Example: {'age': -0.4147618755953326, 'income': 0.14726197944021138, 'loan_amount': 0.4540569498618962, 'credit_score': 1.6493217330729402, 'employment_years': 1.3628553712484424, 'num_prev_loans': 0.0, 'purpose_car': 0.0, 'purpose_education': 1.0, 'purpose_home': 0.0, 'purpose_other': 0.0, 'purpose_unknown': 0.0, 'region_east': 0.0, 'region_north': 1.0, 'region_south': 0.0, 'region_west': 0.0, 'region_unknown': 0.0}


In [37]:
def sigmoid(z: float) -> float:
    return 1 / (1 + math.exp(-max(-500, min(500, z))))


def log_loss(y_true: list[int], y_prob: list[float]) -> float:
    eps = 1e-9
    total = sum(
        -y * math.log(p + eps) - (1 - y) * math.log(1 - p + eps)
        for y, p in zip(y_true, y_prob)
    )
    return total / len(y_true)


def train_logistic(
    X: list[dict],
    y: list[int],
    lr: float = 0.01,
    epochs: int = 100,
) -> dict:
    features = list(X[0].keys())
    weights  = {f: 0.0 for f in features}
    bias     = 0.0

    for epoch in range(epochs):
        for x_row, y_true in zip(X, y):
            z     = sum(weights[f] * x_row[f] for f in features) + bias
            pred  = sigmoid(z)
            error = pred - y_true

            for f in features:
                weights[f] -= lr * error * x_row[f]
            bias -= lr * error

        if epoch % 10 == 0:
            probs = [
                sigmoid(sum(weights[f] * x[f] for f in features) + bias)
                for x in X
            ]
            print(f"Epoch {epoch:3d} | loss: {log_loss(y, probs):.4f}")

    return {"weights": weights, "bias": bias}


def predict_proba(X: list[dict], model: dict) -> list[float]:
    features = list(model["weights"].keys())
    return [
        sigmoid(sum(model["weights"][f] * x[f] for f in features) + model["bias"])
        for x in X
    ]


def predict(X: list[dict], model: dict, threshold: float = 0.5) -> list[int]:
    return [1 if p >= threshold else 0 for p in predict_proba(X, model)]


model      = train_logistic(X_train, y_train, lr=0.01, epochs=150)
val_preds  = predict(X_val, model)
val_probs  = predict_proba(X_val, model)

print("Val predictions sample:", val_preds[:10])
print("Val actual:            ", y_val[:10])
print("Val probabilities:     ", [round(p, 3) for p in val_probs[:10]])

Epoch   0 | loss: 0.4404
Epoch  10 | loss: 0.3712
Epoch  20 | loss: 0.3691
Epoch  30 | loss: 0.3687
Epoch  40 | loss: 0.3685
Epoch  50 | loss: 0.3685
Epoch  60 | loss: 0.3685
Epoch  70 | loss: 0.3685
Epoch  80 | loss: 0.3685
Epoch  90 | loss: 0.3685
Epoch 100 | loss: 0.3685
Epoch 110 | loss: 0.3685
Epoch 120 | loss: 0.3685
Epoch 130 | loss: 0.3685
Epoch 140 | loss: 0.3685
Val predictions sample: [0, 0, 1, 0, 0, 0, 0, 0, 0, 1]
Val actual:             [0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
Val probabilities:      [0.059, 0.067, 0.607, 0.019, 0.162, 0.055, 0.176, 0.161, 0.011, 0.592]
